# Notebook for creating splits for ood experiments.

## Preparing notebook.

### Importing the Libraries.

In [1]:
import scanpy as sc


### Constants.

In [2]:
# working locally?
IS_RUN_LOCALLY = True

# paths
if IS_RUN_LOCALLY:
    H5AD_PATH = "/Users/lorenzo.consoli/Work/data/collab-goettgens/ds_HID01_logicle_100k_UMAP_ann.h5ad"
else:
    H5AD_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/gdrive/ds_HID01_logicle_100k_UMAP_ann.h5ad"

EXPERIMENTAL_COVARIATES = [
    '740-yp_[µm]', 'mtg_[µm]',
    'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]',
    'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]',
    'um729_[µm]', 'scf_[ng_ml]',
    'butyzamide_[nm]', 'retinoic_acid_[µm]',
    'ldl_[ng_ml]', 'il3_[ng_ml]',
    'o2_[%]', "days_of_culture"
]

## Data.

### Reading data.

In [3]:
adata = sc.read_h5ad(H5AD_PATH)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'positives', 'raw'
    

### Annotating with perturbation data.

In [13]:
adata.obs["protocol"] = adata.obs[EXPERIMENTAL_COVARIATES].astype(str).agg("+".join, axis=1)
for idx, protocol in enumerate(adata.obs["protocol"].unique()):
    ood_idxs = adata.obs["protocol"] == protocol
    idistr_idxs = ~ood_idxs
    print(ood_idxs.shape, idistr_idxs.shape)
    break

(100000,) (100000,)


### Defining function for retrieving ood datasets.

In [ ]:
def get_ood_splits(
    adata,
    split_ids, 
    obs_key,
):
    unique_vals = adata.obs[obs_key].unique()
    assert (split_ids < len(unique_vals)).all()
    ood_val = unique_vals[split_ids]
    ood_idxs = adata.obs[obs_key] == ood_val
    train_adata = adata[~ood_idxs].copy()
    ood_adata = adata[ood_idxs].copy()
    return train_adata, ood_adata


def create_protocol_column(
    adata,
    experimental_columns,
    obs_key_added="protocol_id",
    sep="+",
):
    adata.obs[obs_key_added] = adata.obs[experimental_columns].astype(str).agg(sep.join, axis=1)
    return adata


### 

In [ ]:
from collections.abc import Sequence

from anndata import AnnData

def get_split_indices_by_obs_and_unique_value_id(
    adata: AnnData,
    obs_column: str,
    unique_value_ids: Sequence[int]
):
    
    # retrieving unique values of column
    unique_vals = adata.obs[obs_column].unique()
    for val_id in unique_value_ids:
        assert val_id < len(unique_vals)
    ood_vals = unique_vals[unique_value_ids].tolist()
    ood_idxs = adata.obs[obs_column].map(lambda e: e in ood_vals or e == ood_vals).values
    return ~ood_idxs, ood_idxs

def split_adata(adata: AnnData, obs_column: str, unique_value_ids: Sequence[int]):
    """"""

    # get train and ood split indices and slicing the data
    train_idxs, ood_idxs = get_split_indices_by_obs_and_unique_value_id(adata, obs_column, unique_value_ids)
    train_adata = adata[train_idxs].copy()
    ood_adata = adata[ood_idxs].copy()
    return train_adata, ood_adata


In [36]:
split_adata(adata, "protocol", [0, 1, 4, 100])

(100000,) (100000,)


(AnnData object with n_obs × n_vars = 89650 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
     obsm: 'X_pca', 'X_umap'
     layers: 'posit